# T539 Closed-Loop AOS State — converged in-focus images (v1)

**Author:** Aaron Roodman  
**Date Created:** 2026-07-08  
**Last Modified:** 2026-07-08  
**Status:** In Progress  
**Keywords:** Rubin Observatory, AOS, closed loop, MTAOS, wavefront, degree of freedom, trim, v-modes, T539

## Description

Locate the **converged** in-focus image at the end of each BLOCK-T539 closed-loop
alignment run and collect its delivered image quality and AOS state.

Selection:
- `science_program` starts with `BLOCK-T539`
- `observation_reason` == `infocus_initial_alignment`
- observation annotation (`scheduler_note`) is one of:
  `closed_loop_hexapods_10dof_trunc5` or `closed_loop_22dof_trunc12`
- Group contiguous runs (same day, same annotation, seq_num gap ≤ `seq_gap_max`)
  and keep the **last** image of each group — the converged closed-loop image.
- `day_obs` in [`day_obs_min`, `day_obs_max`]

For each kept image we collect:
1. **PSF FWHM median** — ConsDB `visit1_quicklook.psf_sigma_median`
2. **Donut blur FWHM** and **residual AOS FWHM** — ConsDB quicklook (if present)
3. **Aggregated Degree-of-Freedom vector** (total applied Trim / offset from LUT) —
   reuses `aos/code/aos_trim.py`, which anchors on the ConsDB exposure `obs_start`
   and pulls the `MTAOS.logevent_degreeOfFreedom` event in effect before each exposure
4. **Retrieved wavefront (OPD)** per corner WFS — ConsDB `ccdvisit1_quicklook.z4..z28`
   for detectors 191/195/199/203, keeping Z4..Z26 excluding Z20, Z21. These are the
   **total OPD** (not the intrinsic-subtracted deviation) — verified against the
   olr Batoid-based `zk_opd`/`zk_deviation` products (matches OPD to ~0.01 µm,
   deviation off by ~0.11 µm). Same source as olr/code/nightly_table.py. Units: microns.
5. **V-modes** `v1..v12` from the DOF trim, via the OFC sensitivity-matrix SVD on the
   standard_22 DOF set (M2 hex 5 + Cam hex 5 + M1M3 bending 1-7 + M2 bending 1-5),
   12-v-mode scheme, numbered from 1. Normalization = the **OFC config's stored
   `normalization_weights`** (v13), which are the official geom_mean `r^0.5·f^-0.5`
   (field-averaged FWHM; same as `build_ofc_svd` / the bounce analysis). **v1 = focus**
   (M2_dz + Cam_dz). NB: `OFCData()` without `config_dir` loads an old non-geom
   default (v1 becomes M2-tilt, ~1e-4 at convergence) — always pass `config_dir`.
   Also do NOT recompute `sqrt(range/fwhm)` from `compute_normalization_components`:
   that uses corner-point FWHM and is ~√2 off from the config's field-averaged geom.

**Output:** `output/t539_closedloop_aos_{day_obs_min}_{day_obs_max}.parquet`

**Based on:** locate_test_blocks.ipynb (ConsDB), aos/code/aos_trim.py (DOF/Trim),
olr/code/nightly_table.py (per-corner ConsDB Zernikes, obs_start anchoring),
aos/smatrix_vmode_info.ipynb (v-mode SVD)

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-07-08 | Aaron Roodman | Initial version |
| 2026-07-08 | Aaron Roodman | Reuse aos_trim for DOF; fix ConsDB query to use v1.* |

## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Helper Functions](#functions)
4. [Data Access](#data)
5. [Analysis](#analysis)
6. [Results & Plots](#results)

<a id='params'></a>
## Parameters

In [ ]:
# ============================================================
# Parameters — All configurable values collected here
# ============================================================

day_obs_min = 20260420
day_obs_max = 20260708          # bump to today when re-running

instrument = 'lsstcam'
science_program_prefix = 'BLOCK-T539'
observation_reason = 'infocus_initial_alignment'

# Closed-loop convergence annotations (scheduler_note) of interest. We keep
# the LAST in-focus image of each contiguous run with one of these — i.e.
# the converged image at the end of a closed-loop alignment sequence.
annotations = [
    'closed_loop_hexapods_10dof_trunc5',
    'closed_loop_22dof_trunc12',
]
seq_gap_max = 10   # max seq_num gap that still counts as the same contiguous run

consdb_url = 'http://consdb-pq.consdb:8080/consdb'

# OFC config dir for the sensitivity matrix / v-mode SVD (v13 = telescope AOS)
ofc_config_dir = '/home/r/roodman/u/LSST/packages/ts_config_mttcs/MTAOS/v13/ofc'

# Corner wavefront sensors (SW0 half-chips): detector id -> raft name
corners = {191: 'R00_SW0', 195: 'R04_SW0', 199: 'R40_SW0', 203: 'R44_SW0'}

# Zernikes to keep: Z4..Z26 excluding Z20, Z21 (Noll) — 21 coefficients
zk_noll = [z for z in range(4, 27) if z not in (20, 21)]

output_dir = 'output'

<a id='setup'></a>
## Setup & Imports

In [ ]:
import os
import sys
from pathlib import Path

os.environ["no_proxy"] += ",.consdb"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from astropy.time import Time
from astropy.table import Table

from tqdm.notebook import tqdm

# Add repo root to path for common imports
sys.path.insert(0, str(Path.cwd().parent))
from common.utils import setup_plotting
setup_plotting()

# Shared AOS helpers (aos/code): DOF trim, geom v-modes, per-corner Zernikes
sys.path.insert(0, str(Path.cwd().parent / 'aos' / 'code'))
import aos_trim
import aos_state

<a id='functions'></a>
## Helper Functions

In [ ]:
# Per-visit AOS-state helpers are shared in aos/code/aos_state.py (imported above):
#   aos_state.fetch_corner_zernikes_consdb(client, visit_ids)  -> per-corner OPD
#   aos_state.build_geom_svd(config_dir, dof_set)              -> geom SVD
#   aos_state.project_dofs_to_vmodes(dof_state, svd, n_modes)  -> v-modes
# and DOF trim comes from aos_trim.fetch_aggregated_dof_for_visits.
# No notebook-local helpers needed.

<a id='data'></a>
## Data Access

In [ ]:
# Select images from ConsDB. Query the day_obs range only (robust — matches
# locate_test_blocks), then filter and group in pandas.
client = aos_trim.make_consdb_client(consdb_url)

visits_query = f'''
    SELECT v1.*,
           ql.physical_rotator_angle,
           ql.psf_sigma_median,
           ql.seeing_zenith_500nm_median
    FROM cdb_{instrument}.visit1 v1
    LEFT JOIN cdb_{instrument}.visit1_quicklook ql
      ON v1.visit_id = ql.visit_id
    WHERE v1.day_obs >= {day_obs_min} AND v1.day_obs <= {day_obs_max}
'''
allvisits = client.query(visits_query).to_pandas()
print(f"Fetched {len(allvisits)} visits in day_obs range")

# Candidate images: T539, infocus_initial_alignment, annotation in our set
sp = allvisits['science_program'].fillna('')
note = allvisits['scheduler_note'].fillna('')
cand = allvisits[
    sp.str.startswith(science_program_prefix)
    & (allvisits['observation_reason'] == observation_reason)
    & note.isin(annotations)
].copy()
cand = cand.sort_values(['day_obs', 'seq_num']).reset_index(drop=True)
print(f"Candidate infocus images (either annotation): {len(cand)}")

# Contiguous closed-loop runs: a new group starts when the day changes, the
# annotation changes, or seq_num jumps by more than seq_gap_max. Keep the
# LAST image of each group = the converged closed-loop image.
grp = (
    (cand['day_obs'] != cand['day_obs'].shift())
    | (cand['scheduler_note'] != cand['scheduler_note'].shift())
    | ((cand['seq_num'] - cand['seq_num'].shift()) > seq_gap_max)
).cumsum()
cand['group_id'] = grp
grp_size = cand.groupby('group_id').size()

visits = cand.groupby('group_id').tail(1).copy()
visits['n_in_group'] = visits['group_id'].map(grp_size).values
visits = visits.sort_values(['day_obs', 'seq_num']).reset_index(drop=True)

# PSF FWHM median (arcsec) from psf_sigma_median (pixels, 0.2"/pix)
visits['psf_fwhm_median'] = 2.355 * pd.to_numeric(
    visits['psf_sigma_median'], errors='coerce') * 0.2

print(f"Contiguous groups (kept last image each): {len(visits)}")
print(visits['scheduler_note'].value_counts().to_string())
visits[['day_obs', 'seq_num', 'band', 'scheduler_note',
        'n_in_group', 'psf_fwhm_median']].head(30)

In [ ]:
# Donut blur + residual AOS FWHM (may or may not be in the quicklook schema)
if len(visits) > 0:
    ids = ",".join(str(v) for v in visits['visit_id'])
    for col in ('donut_blur_fwhm', 'aos_fwhm'):
        try:
            q = f'''SELECT visit_id, {col} FROM cdb_{instrument}.visit1_quicklook
                    WHERE visit_id IN ({ids})'''
            extra = client.query(q).to_pandas()
            visits = visits.merge(extra, on='visit_id', how='left')
            print(f"Added {col}")
        except Exception as e:
            print(f"{col} not available in ConsDB: {type(e).__name__}")

<a id='analysis'></a>
## Analysis

In [ ]:
# Aggregated DOF (total applied Trim / offset from LUT), reusing aos_trim.
# It anchors on the ConsDB exposure obs_start and takes the degreeOfFreedom
# event in effect just before each exposure (same approach as nightly_table.py).
#
# Construct EfdClient directly with output_mode="dataframe"; makeEfdClient()
# can raise "Invalid output format" on the USDF 13.0 stack.
from lsst_efd_client import EfdClient
efd = EfdClient("usdf_efd", output_mode="dataframe")

fit_table = Table.from_pandas(visits[['day_obs', 'seq_num']].astype(int))
trim, dof_info = aos_trim.fetch_aggregated_dof_for_visits(
    fit_table, efd_client=efd, consdb_client=client)
print(f"DOF anchored on obs_start: {dof_info['n_obs_start']}, "
      f"with finite result: {dof_info['n_dof']} / {len(visits)}")

for i in range(aos_trim.N_DOF):
    visits[f'dof{i}'] = trim[:, i]

In [ ]:
# V-modes from the DOF trim via the OFC StateEstimator (standard_22, 12 modes),
# geom-normalized (config weights). Uses the shared aos_state helper so these
# are identical to the OLR nightly_table and the nightly report (all call
# get_vmodes_from_dofs on the same Vh). v1 = focus. Numbered from 1.
#
# Note: degenerate singular-value pairs (e.g. astigmatism/decenter doublets)
# make the *individual* v-modes in a pair basis-dependent; only v1 and the
# per-pair magnitudes are unique. Sharing StateEstimator's Vh keeps all code
# consistent regardless.
N_VMODE = 12
se = aos_state.make_state_estimator(config_dir=ofc_config_dir,
                                    dof_set='standard_22')
dof_cols = [f'dof{i}' for i in range(aos_trim.N_DOF)]
dof_mat = visits[dof_cols].to_numpy(dtype=float)          # (n_visits, 50) DOF trim
vmodes = aos_state.vmodes_from_dofs(dof_mat, se, n_modes=N_VMODE)

for j in range(N_VMODE):
    visits[f'v{j + 1}'] = vmodes[:, j]

print(f"Computed v1..v{N_VMODE} (StateEstimator, geom) for "
      f"{int(np.isfinite(vmodes).all(axis=1).sum())}/{len(visits)} visits")
print("median |v_j|:", np.round(np.nanmedian(np.abs(vmodes), axis=0), 4))

In [ ]:
# Retrieved wavefront (OPD) per corner, from ConsDB ccdvisit1_quicklook
# (shared helper; already associated with each visit).
zk_df = aos_state.fetch_corner_zernikes_consdb(
    client, visits['visit_id'].values, instrument=instrument,
    zk_noll=zk_noll, corners=corners)
visits = visits.merge(zk_df, left_on='visit_id', right_index=True, how='left')

zcols = [c for c in visits.columns if c.startswith('z') and '_SW0' in c]
n_with = int(visits[zcols].notna().any(axis=1).sum()) if zcols else 0
print(f"Added {len(zcols)} per-corner Zernike columns "
      f"({len(zk_noll)} Zernikes x {len(corners)} corners); "
      f"{n_with}/{len(visits)} visits with wavefront data")

<a id='results'></a>
## Results & Plots

In [ ]:
# Summary of delivered image quality per converged image
show = ['day_obs', 'seq_num', 'band', 'scheduler_note', 'n_in_group',
        'psf_fwhm_median', 'donut_blur_fwhm', 'aos_fwhm',
        'altitude', 'physical_rotator_angle']
show = [c for c in show if c in visits.columns]
visits[show]

In [ ]:
# Write full table (metadata + IQ + DOF + per-corner Zernikes) to parquet
os.makedirs(output_dir, exist_ok=True)
out_file = os.path.join(
    output_dir, f't539_closedloop_aos_{day_obs_min}_{day_obs_max}.parquet')
visits.to_parquet(out_file, index=False)
print(f"Wrote {len(visits)} rows x {visits.shape[1]} cols to {out_file}")